# 03 — BERTopic Baseline

BERTopic is an end-to-end topic model that wraps sentence-transformers + UMAP + HDBSCAN.
We run it as a **baseline**: every custom-pipeline result in notebooks 04–06 is compared against the best BERTopic configuration found here.

## Experiments

| # | What varies | Purpose |
|---|---|---|
| B1 | Default BERTopic (all defaults) | True out-of-the-box baseline |
| B2 | Custom embedding: `all-mpnet-base-v2` | Better base model, same internals |
| B3 | Custom embedding: `instructor-large` + domain instruction | Instruction-tuned embeddings via BERTopic |
| B4 | `min_topic_size` sweep [5, 10, 20] | Granularity vs. noise trade-off |

All results are logged to `data/cache/results.csv` via `log_result()`.
The internal UMAP embedding is extracted for silhouette computation so BERTopic rows are directly comparable to custom-pipeline rows.

**Prerequisite:** Run `02_business_data_exploration.ipynb` first to generate `sample_hotels_5k.jsonl`.

## Setup

In [1]:
import sys, json, time
import numpy as np
sys.path.insert(0, '..')

from utils import (
    load_config, get_preprocessor,
    embedding_path, save_array, load_array, array_exists,
    compute_metrics, log_result,
)

cfg = load_config(sample_size=5_000, device="cpu")
cfg.ensure_dirs()
print(cfg)

PipelineConfig(
  sample_size  = 5,000
  data_file    = sample_hotels_10k.jsonl
  device       = cpu
  batch_size   = 64
  seed         = 42
  cache_dir    = /home/dominik/Documents/Informatik/4_Semester/NLP_Projekt/ReviewScope/data/cache
)


In [2]:
from bertopic import BERTopic

In [3]:
# Load reviews
preprocess = get_preprocessor("minimal")
texts = []
with open(cfg.data_path) as f:
    for line in f:
        obj = json.loads(line)
        texts.append(preprocess(obj["text"]))
        if len(texts) >= cfg.sample_size:
            break

print(f"Loaded {len(texts):,} reviews  |  example: {texts[0][:80]!r}")

Loaded 5,000 reviews  |  example: 'We checked in around 2:30 pm. Check-in was quick and easy with complimentary val'


## B1 — Default BERTopic

In [4]:
from bertopic import BERTopic

model_dir = cfg.bertopic_dir / f"default__{cfg.sample_size // 1000}k"

if model_dir.exists():
    print(f"[cache hit] Loading saved BERTopic model from {model_dir}")
    topic_model_b1 = BERTopic.load(str(model_dir))
    doc_info = topic_model_b1.get_document_info(texts)
    labels_b1 = doc_info["Topic"].to_numpy()
    umap_b1 = topic_model_b1.umap_model.embedding_
    runtime_b1 = 0.0  # already computed
else:
    t0 = time.time()
    topic_model_b1 = BERTopic(
        verbose=True,
        calculate_probabilities=False,
    )
    topics, _ = topic_model_b1.fit_transform(texts)
    runtime_b1 = time.time() - t0
    labels_b1 = np.array(topics)
    umap_b1 = topic_model_b1.umap_model.embedding_
    topic_model_b1.save(str(model_dir))
    print(f"Saved to {model_dir}")

print(f"\nTopics found: {topic_model_b1.get_topic_info().shape[0] - 1}  (excluding -1)")
print(f"Runtime: {runtime_b1:.1f}s")

metrics_b1 = compute_metrics(umap_b1, labels_b1, runtime_s=runtime_b1, seed=cfg.seed)
print(metrics_b1)

[cache hit] Loading saved BERTopic model from /home/dominik/Documents/Informatik/4_Semester/NLP_Projekt/ReviewScope/data/cache/bertopic/default__5k

Topics found: 70  (excluding -1)
Runtime: 0.0s
{'n_docs': 5000, 'n_clusters': 70, 'noise_count': 1927, 'noise_ratio': 0.3854, 'silhouette': 0.6144, 'davies_bouldin': 0.4943, 'calinski_harabasz': 3693.4, 'runtime_s': 0.0}


In [5]:
log_result(cfg.results_csv, {
    "pipeline": "bertopic",
    "sample_size": cfg.sample_size,
    "device": cfg.device,
    "embedding_model": "all-MiniLM-L6-v2",   # BERTopic default
    "embedding_instruction": "no_inst",
    "embed_dim": 384,
    "reduction_method": "umap",
    "umap_n_components": 5,    # BERTopic default
    "umap_n_neighbors": 15,
    "umap_min_dist": 0.0,
    "umap_metric": "cosine",
    "clustering_algo": "bertopic_internal",
    "cluster_params": json.dumps({"min_topic_size": 10, "top_n_words": 10}),
    **metrics_b1,
    "notes": "BERTopic defaults",
})

  [skip]   run_id=d3e7a7b5  (already logged)


In [6]:
# Top 10 topics
topic_model_b1.get_topic_info().head(12)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1927,-1_the_and_to_was,"[the, and, to, was, in, for, room, we, of, it]","[I stayed her for a business trip, and the roo..."
1,0,314,0_oysters_shrimp_food_was,"[oysters, shrimp, food, was, and, good, the, w...",[The reason that this restaurant is getting 5 ...
2,1,151,1_orleans_new_the_is,"[orleans, new, the, is, and, of, in, french, t...",[A last minute vacation turned into a wonderfu...
3,2,141,2_to_me_that_my,"[to, me, that, my, she, told, they, we, the, for]",[I booked this hotel for myself and my boyfrie...
4,3,133,3_hotel_great_and_is,"[hotel, great, and, is, very, staff, location,...",[This hotel is wonderful. My husband and I hav...
5,4,109,4_philly_philadelphia_the_in,"[philly, philadelphia, the, in, and, hotel, wa...",[This hotel is a gem in the heart of Philadelp...
6,5,102,5_bar_drinks_drink_and,"[bar, drinks, drink, and, of, atmosphere, you,...",[I truly enjoyed my first experience at Plat 9...
7,6,96,6_nashville_and_to_the,"[nashville, and, to, the, we, hotel, was, in, ...","[First things first...This hotel is very big, ..."
8,7,91,7_noise_loud_the_hear,"[noise, loud, the, hear, hotel, was, night, is...","[Everything in the room is clean and ""moderate..."
9,8,89,8_french_quarter_hotel_the,"[french, quarter, hotel, the, is, in, and, str...",[This wouldn't be considered a 4-star hotel in...


## B2 — Custom embedding: `all-mpnet-base-v2` (no instruction)

In [5]:
from sentence_transformers import SentenceTransformer

MODEL_B2 = "all-mpnet-base-v2"
emb_path_b2 = embedding_path(cfg.cache_dir, MODEL_B2, cfg.sample_size, instruction="no_inst")

if array_exists(emb_path_b2):
    embeddings_b2 = load_array(emb_path_b2)
    embed_time_b2 = 0.0
else:
    t0 = time.time()
    st = SentenceTransformer(MODEL_B2, device=cfg.device)
    embeddings_b2 = st.encode(texts, batch_size=cfg.batch_size, show_progress_bar=True,
                              convert_to_numpy=True)
    embed_time_b2 = time.time() - t0
    save_array(emb_path_b2, embeddings_b2)

print(f"Embeddings shape: {embeddings_b2.shape}")

  [loaded] all-mpnet-base-v2__no_inst__5k.npy  shape=(5000, 768)  dtype=float32
Embeddings shape: (5000, 768)


In [6]:
model_dir_b2 = cfg.bertopic_dir / f"mpnet_no_inst__{cfg.sample_size // 1000}k"

if model_dir_b2.exists():
    print(f"[cache hit] {model_dir_b2}")
    topic_model_b2 = BERTopic.load(str(model_dir_b2))
    doc_info = topic_model_b2.get_document_info(texts)
    labels_b2 = doc_info["Topic"].to_numpy()
    umap_b2 = topic_model_b2.umap_model.embedding_
    runtime_b2 = 0.0
else:
    t0 = time.time()
    topic_model_b2 = BERTopic(verbose=True, calculate_probabilities=False)
    topics, _ = topic_model_b2.fit_transform(texts, embeddings=embeddings_b2)
    runtime_b2 = embed_time_b2 + (time.time() - t0)
    labels_b2 = np.array(topics)
    umap_b2 = topic_model_b2.umap_model.embedding_
    topic_model_b2.save(str(model_dir_b2))

metrics_b2 = compute_metrics(umap_b2, labels_b2, runtime_s=runtime_b2, seed=cfg.seed)
print(metrics_b2)

log_result(cfg.results_csv, {
    "pipeline": "bertopic",
    "sample_size": cfg.sample_size,
    "device": cfg.device,
    "embedding_model": MODEL_B2,
    "embedding_instruction": "no_inst",
    "embed_dim": 768,
    "embed_time_s": embed_time_b2,
    "reduction_method": "umap",
    "umap_n_components": 5,
    "umap_n_neighbors": 15,
    "umap_min_dist": 0.0,
    "umap_metric": "cosine",
    "clustering_algo": "bertopic_internal",
    "cluster_params": json.dumps({"min_topic_size": 10, "top_n_words": 10}),
    **metrics_b2,
    "notes": "BERTopic + all-mpnet-base-v2",
})

[cache hit] /home/dominik/Documents/Informatik/4_Semester/NLP_Projekt/ReviewScope/data/cache/bertopic/mpnet_no_inst__5k
{'n_docs': 5000, 'n_clusters': 59, 'noise_count': 1557, 'noise_ratio': 0.3114, 'silhouette': 0.5514, 'davies_bouldin': 0.4941, 'calinski_harabasz': 4683.6, 'runtime_s': 0.0}
  [skip]   run_id=db15c1f4  (already logged)


## B3 — Custom embedding: `instructor-large` + domain instruction

In [9]:
# Load instructor-large directly via sentence-transformers 3.x — no InstructorEmbedding package needed.
# sentence-transformers 3.x supports INSTRUCTOR models natively via the prompt= parameter,
# which prepends the instruction string to each text before encoding (same behaviour as the
# original InstructorEmbedding [instruction, text] pair format).
from sentence_transformers import SentenceTransformer

MODEL_B3   = "hkunlp/instructor-large"
INSTR_B3   = "domain"
INSTR_TEXT = "Represent the hotel review for clustering by theme (room quality, staff, location, breakfast, cleanliness, value):"

emb_path_b3 = embedding_path(cfg.cache_dir, MODEL_B3, cfg.sample_size, instruction=INSTR_B3)

if array_exists(emb_path_b3):
    embeddings_b3 = load_array(emb_path_b3)
    embed_time_b3 = 0.0
else:
    t0 = time.time()
    st_instr = SentenceTransformer(MODEL_B3, device=cfg.device)
    embeddings_b3 = st_instr.encode(
        texts,
        prompt=INSTR_TEXT,
        batch_size=cfg.batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    embed_time_b3 = time.time() - t0
    save_array(emb_path_b3, embeddings_b3)
    print(f"Encoded in {embed_time_b3:.1f}s")

print(f"embeddings_b3 shape: {embeddings_b3.shape}")


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
if embeddings_b3 is not None:
    model_dir_b3 = cfg.bertopic_dir / f"instructor_large_domain__{cfg.sample_size // 1000}k"

    if model_dir_b3.exists():
        print(f"[cache hit] {model_dir_b3}")
        topic_model_b3 = BERTopic.load(str(model_dir_b3))
        doc_info = topic_model_b3.get_document_info(texts)
        labels_b3 = doc_info["Topic"].to_numpy()
        umap_b3 = topic_model_b3.umap_model.embedding_
        runtime_b3 = 0.0
    else:
        t0 = time.time()
        topic_model_b3 = BERTopic(verbose=True, calculate_probabilities=False)
        topics, _ = topic_model_b3.fit_transform(texts, embeddings=embeddings_b3)
        runtime_b3 = embed_time_b3 + (time.time() - t0)
        labels_b3 = np.array(topics)
        umap_b3 = topic_model_b3.umap_model.embedding_
        topic_model_b3.save(str(model_dir_b3))

    metrics_b3 = compute_metrics(umap_b3, labels_b3, runtime_s=runtime_b3, seed=cfg.seed)
    print(metrics_b3)

    log_result(cfg.results_csv, {
        "pipeline": "bertopic",
        "sample_size": cfg.sample_size,
        "device": cfg.device,
        "embedding_model": MODEL_B3,
        "embedding_instruction": INSTR_B3,
        "embed_dim": 768,
        "embed_time_s": embed_time_b3,
        "reduction_method": "umap",
        "umap_n_components": 5,
        "umap_n_neighbors": 15,
        "umap_min_dist": 0.0,
        "umap_metric": "cosine",
        "clustering_algo": "bertopic_internal",
        "cluster_params": json.dumps({"min_topic_size": 10, "top_n_words": 10}),
        **metrics_b3,
        "notes": f"BERTopic + {MODEL_B3} + domain instruction",
    })

## B4 — `min_topic_size` sweep

In [ ]:
import pandas as pd

sweep_results = []

for mts in [5, 10, 20]:
    model_dir_sweep = cfg.bertopic_dir / f"default_mts{mts}__{cfg.sample_size // 1000}k"

    if model_dir_sweep.exists():
        print(f"[cache hit] min_topic_size={mts}")
        tm = BERTopic.load(str(model_dir_sweep))
        doc_info = tm.get_document_info(texts)
        labels_s = doc_info["Topic"].to_numpy()
        umap_s = tm.umap_model.embedding_
        runtime_s = 0.0
    else:
        t0 = time.time()
        tm = BERTopic(min_topic_size=mts, verbose=False, calculate_probabilities=False)
        topics, _ = tm.fit_transform(texts)
        runtime_s_val = time.time() - t0
        labels_s = np.array(topics)
        umap_s = tm.umap_model.embedding_
        tm.save(str(model_dir_sweep))

    mets = compute_metrics(umap_s, labels_s, runtime_s=runtime_s_val if not model_dir_sweep.exists() else 0.0,
                           seed=cfg.seed)
    sweep_results.append({"min_topic_size": mts, **mets})
    print(f"  min_topic_size={mts}  clusters={mets['n_clusters']}  "
          f"silhouette={mets['silhouette']}  noise={mets['noise_ratio']:.1%}")

    log_result(cfg.results_csv, {
        "pipeline": "bertopic",
        "sample_size": cfg.sample_size,
        "device": cfg.device,
        "embedding_model": "all-MiniLM-L6-v2",
        "embedding_instruction": "no_inst",
        "embed_dim": 384,
        "reduction_method": "umap",
        "umap_n_components": 5,
        "umap_n_neighbors": 15,
        "umap_min_dist": 0.0,
        "umap_metric": "cosine",
        "clustering_algo": "bertopic_internal",
        "cluster_params": json.dumps({"min_topic_size": mts, "top_n_words": 10}),
        **mets,
        "notes": f"BERTopic min_topic_size sweep mts={mts}",
    })

pd.DataFrame(sweep_results).set_index("min_topic_size")

## BERTopic topic word inspection

Quick look at what the default model found — useful for sanity-checking semantic coherence before comparing metrics.

In [ ]:
info = topic_model_b1.get_topic_info()
print(f"Total topics (incl. outlier): {len(info)}")
print(info[["Topic", "Count", "Name"]].head(15).to_string(index=False))

print("\nTop words per topic (first 8):")
for tid in info["Topic"].values[1:9]:   # skip -1 (outlier)
    words = [w for w, _ in topic_model_b1.get_topic(tid)[:6]]
    print(f"  Topic {tid:>3d}: {', '.join(words)}")

## Findings

| Experiment | n_clusters | noise_ratio | silhouette | notes |
|---|---|---|---|---|
| B1 — defaults | _fill_ | _fill_ | _fill_ | |
| B2 — mpnet | _fill_ | _fill_ | _fill_ | |
| B3 — instructor-large domain | _fill_ | _fill_ | _fill_ | |
| B4 — mts=5 | _fill_ | _fill_ | _fill_ | |
| B4 — mts=10 | _fill_ | _fill_ | _fill_ | |
| B4 — mts=20 | _fill_ | _fill_ | _fill_ | |

**Best BERTopic configuration:** _fill in after running_

**Baseline silhouette to beat (custom pipeline):** _fill in_